# AeroRecover AI — Veri Keşfi

Bu notebook, BTS On-Time Performance verisini yükler ve temel keşifsel veri analizini gerçekleştirir.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_flights import load_flights

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## Veriyi yükleme

In [ ]:
flights = load_flights()
print(f'Satır sayısı: {len(flights):,}')
print(f'Sütun sayısı: {flights.shape[1]}')

## İlk 5 satır

In [ ]:
flights.head()

## Veri tipleri

In [ ]:
flights.dtypes.to_frame(name='veri_tipi')

## Eksik değerler

In [ ]:
missing_values = pd.DataFrame({
    'eksik_deger': flights.isna().sum(),
    'eksik_oran_yuzde': flights.isna().mean().mul(100).round(2),
}).sort_values('eksik_deger', ascending=False)
missing_values[missing_values['eksik_deger'] > 0]

## Gecikme sütunlarının dağılımı

In [ ]:
columns_by_upper_name = {column.upper(): column for column in flights.columns}
delay_names = [
    'DEP_DELAY', 'ARR_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY',
    'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
]
delay_columns = [columns_by_upper_name[name] for name in delay_names if name in columns_by_upper_name]

if not delay_columns:
    raise KeyError('BTS gecikme sütunlarından hiçbiri veri setinde bulunamadı.')

flights[delay_columns].describe().T

## En fazla geciken ilk 10 havalimanı

In [ ]:
origin_column = columns_by_upper_name.get('ORIGIN')
arrival_delay_column = columns_by_upper_name.get('ARR_DELAY')

if origin_column is None or arrival_delay_column is None:
    raise KeyError('Havalimanı analizi için ORIGIN ve ARR_DELAY sütunları gereklidir.')

top_delayed_airports = (
    flights.groupby(origin_column)[arrival_delay_column]
    .agg(ortalama_gecikme='mean', ucus_sayisi='count')
    .sort_values('ortalama_gecikme', ascending=False)
    .head(10)
)
top_delayed_airports

In [ ]:
ax = top_delayed_airports['ortalama_gecikme'].sort_values().plot(
    kind='barh', figsize=(10, 6), color='tomato'
)
ax.set_title('Ortalama Varış Gecikmesine Göre İlk 10 Kalkış Havalimanı')
ax.set_xlabel('Ortalama varış gecikmesi (dakika)')
ax.set_ylabel('Havalimanı')
plt.tight_layout()
plt.show()

## Histogram grafikleri

In [ ]:
flights[delay_columns].hist(bins=50, figsize=(16, 10), color='steelblue', edgecolor='white')
plt.suptitle('Gecikme Sütunlarının Histogramları', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## Boxplot grafikleri

In [ ]:
plt.figure(figsize=(14, 7))
sns.boxplot(data=flights[delay_columns], orient='h', showfliers=False)
plt.title('Gecikme Sütunlarının Boxplot Grafikleri')
plt.xlabel('Gecikme (dakika)')
plt.ylabel('Gecikme türü')
plt.tight_layout()
plt.show()